# Requirements check — opsly-assgn

Run this notebook **before** implementing the MCP multi-agent system.

**Setup (once):**
1. `python3.12 -m venv .venv` and activate
2. `pip install -e ".[dev]"` (includes Jupyter)
3. `./scripts/bootstrap_env.sh` to create `.env` from `keys.txt`
4. In Cursor/VS Code: select kernel **Python 3.12 (.venv)**

Secrets are never printed — only masked values.

**Neo4j Aura `Unable to retrieve routing information`:** instance is usually **paused** — open [Neo4j Aura Console](https://console.neo4j.io) → **Resume**. Username is typically `neo4j` (not the instance id). You can still **READY TO BUILD** with a WARN on Neo4j and use Docker Neo4j during implementation.

In [ ]:
import sys
from pathlib import Path

# Works whether the kernel cwd is `opsly-assgn/` or `opsly-assgn/notebooks/`
ROOT = Path.cwd().resolve()
for candidate in (ROOT, ROOT.parent, *ROOT.parents):
    if (candidate / "pyproject.toml").is_file():
        ROOT = candidate
        break

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Python:", sys.executable)
print("Expected venv:", ROOT / ".venv" / "bin" / "python")
if ".venv" not in sys.executable:
    print("WARNING: not using project .venv — Select Kernel → Python 3.12 (opsly-assgn .venv)")

In [ ]:
from opsly.env_loader import load_project_env
from opsly.preflight import discover_root, ready_to_build, run_all_checks, summarize

root = discover_root(ROOT)
env_source = load_project_env(root)
print("Config loaded from:", env_source)
results = run_all_checks(root)
counts = summarize(results)
counts

In [ ]:
rows = [
    {"category": r.category, "check": r.name, "status": r.status, "detail": r.detail}
    for r in results
]
status_order = {"fail": 0, "warn": 1, "skip": 2, "pass": 3}
rows.sort(key=lambda row: status_order.get(row["status"], 99))

from IPython.display import display

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    try:
        import jinja2  # noqa: F401 — required for df.style

        def color_status(val: str) -> str:
            colors = {"pass": "#d4edda", "warn": "#fff3cd", "fail": "#f8d7da", "skip": "#e2e3e5"}
            return f"background-color: {colors.get(val, 'white')}"

        styler = df.style
        if hasattr(styler, "map"):
            styler = styler.map(color_status, subset=["status"])
        else:
            styler = styler.applymap(color_status, subset=["status"])
        display(styler)
    except (ImportError, AttributeError):
        print("Tip: pip install jinja2 for colored table (optional)")
        display(df)
except ImportError:
    print("Install pandas: .venv/bin/pip install pandas")
    for row in rows:
        print(f"{row['status']:4} [{row['category']}] {row['check']}: {row['detail']}")

In [ ]:
failures = [r for r in results if r.status == "fail"]
warnings = [r for r in results if r.status == "warn"]

print("=" * 60)
if ready_to_build(results):
    print("READY TO BUILD: environment and credentials look good.")
    print("Implementation rows marked 'skip' are expected until we scaffold agents.")
else:
    print("NOT READY: fix FAIL items below before starting implementation.")
print("=" * 60)

if failures:
    print("\nFAIL:")
    for r in failures:
        print(f"  [{r.category}] {r.name}: {r.detail}")

if warnings:
    print("\nWARN:")
    for r in warnings:
        print(f"  [{r.category}] {r.name}: {r.detail}")

## Assignment checklist (manual — from `ass.md` / `HOW_TO_USE.md`)

After preflight passes, the **build phase** must deliver:

| Area | Deliverable |
|------|-------------|
| MCP | 5 servers: Orchestrator, Indexer, Graph Query, Code Analyst, Memory |
| Graph | Neo4j schema + indexer; optional seed for reviewers |
| API | Gateway: `/api/chat`, `/api/index`, health, stats, WebSocket |
| Ops | Docker Compose, per-service Dockerfiles, `.env.example`, tests >70% |
| Docs | README architecture + walkthrough |

Optional Groq model from your `keys.txt` sample: set `GROQ_MODEL_SYNTHESIS=qwen/qwen3-32b` in `.env` if you prefer Qwen over Llama defaults.

In [ ]:
# Optional: quick Groq streaming smoke test (same pattern as keys.txt sample)
import os

from opsly.env_loader import load_project_env

load_project_env(root)
key = os.getenv("GROQ_API_KEY", "").strip()
if not key:
    print("Skip streaming test — no GROQ_API_KEY")
else:
    from groq import Groq

    from opsly.env_loader import groq_models_to_try

    client = Groq(api_key=key)
    for model in groq_models_to_try("synthesis"):
        try:
            stream = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": "Say hello in five words."}],
                max_tokens=32,
                stream=True,
            )
            chunks = []
            for chunk in stream:
                chunks.append(chunk.choices[0].delta.content or "")
            print("Model:", model)
            print("Streamed:", "".join(chunks).strip())
            break
        except Exception as exc:
            print(f"Failed {model}:", type(exc).__name__)
    else:
        print("All synthesis models failed — set GROQ_MODEL_SYNTHESIS in .env")